In [1]:
import torch, sentencepiece as spm, json, re
from pathlib import Path

CKPT_PATH = "checkpoints/best_step8000_val3.862.pt"
SPM_PATH  = "tokenizer/ka_sp_24000.model"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Using:", DEVICE)


Using: cuda


In [2]:
from src.models import GPT, GPTConfig

ckpt = torch.load(CKPT_PATH, map_location="cpu")
cfg = GPTConfig(**ckpt["config"])
model = GPT(cfg).to(DEVICE)
model.load_state_dict(ckpt["model"], strict=True)
model.eval()

sp = spm.SentencePieceProcessor(model_file=SPM_PATH)
BOS, EOS, UNK = sp.bos_id(), sp.eos_id(), sp.unk_id()

print("Model:", cfg)
print("Vocab:", sp.vocab_size(), "UNK:", UNK)


Model: GPTConfig(vocab_size=24000, n_layer=12, n_head=8, d_model=512, n_ctx=512, ffn_mult=4, dropout=0.0, rope_theta=10000.0, tie_weights=True, bias=False, use_sdpa=True, grad_checkpoint=False)
Vocab: 24000 UNK: 0


In [3]:
GEORGIAN_ONLY_DECODE = True
GE_PIECE_RE = re.compile(r"^▁?[ა-ჰ]+(?:[-– ][ა-ჰ]+)?$")

def build_ge_mask():
    mask = torch.full((sp.vocab_size(),), float("-inf"))
    space_id = sp.piece_to_id("▁")
    if space_id >= 0: mask[space_id] = 0.0
    for tid in range(sp.vocab_size()):
        if GE_PIECE_RE.match(sp.id_to_piece(tid)):
            mask[tid] = 0.0
    if EOS is not None and EOS >= 0: mask[EOS] = 0.0
    return mask.to(DEVICE)

GE_MASK = build_ge_mask() if GEORGIAN_ONLY_DECODE else None


In [4]:
@torch.no_grad()
def generate(
    prompt: str,
    max_new_tokens=12, min_new_tokens=2,
    temperature=0.1, top_p=0.9, top_k=0,
    rep_penalty=1.20, no_repeat_ngram_size=4,
    stop_on_eos=False, stop_at_punct=True, add_bos=True,
):
    ids = sp.encode(prompt, out_type=int, add_bos=add_bos, add_eos=False)
    x = torch.tensor(ids, dtype=torch.long, device=DEVICE)[None, :]
    start = x.size(1)
    seen = {}
    if no_repeat_ngram_size > 0 and x.size(1) >= no_repeat_ngram_size:
        for i in range(x.size(1) - no_repeat_ngram_size + 1):
            gram = tuple(x[0, i:i+no_repeat_ngram_size].tolist())
            seen[gram] = seen.get(gram, 0) + 1

    def update_seen(seq):
        n = no_repeat_ngram_size
        if n <= 0 or seq.size(1) < n: return
        gram = tuple(seq[0, -n:].tolist())
        seen[gram] = seen.get(gram, 0) + 1

    def violates(next_id):
        n = no_repeat_ngram_size
        if n <= 0 or x.size(1) < n - 1: return False
        tail = tuple(x[0, -(n-1):].tolist()) if n > 1 else tuple()
        return seen.get(tail + (int(next_id.item()),), 0) > 0

    PUNCTS = [".","!","?","…","։","።"]

    for t in range(max_new_tokens):
        logits, _ = model(x)
        logits = logits[:, -1, :]

        if UNK is not None and UNK >= 0:
            logits[:, UNK] = -1e9
        if GE_MASK is not None:
            logits = logits + GE_MASK

        for tok in ["▁ეს","▁არის","▁და","▁რომ"]:
            tid = sp.piece_to_id(tok)
            if tid >= 0: logits[:, tid] /= rep_penalty
        if rep_penalty != 1.0:
            for tid in set(x[0].tolist()):
                logits[:, tid] /= rep_penalty

        if temperature <= 1e-6:
            next_id = logits.argmax(dim=-1)
        else:
            logits = logits / temperature
            if top_k > 0:
                v, ix = torch.topk(logits, top_k)
                keep = torch.full_like(logits, -float("inf"))
                keep.scatter_(1, ix, v); logits = keep
            if top_p < 1.0:
                sl, si = torch.sort(logits, descending=True)
                p = torch.softmax(sl, dim=-1)
                cp = torch.cumsum(p, dim=-1)
                cut = (cp > top_p).float().argmax(dim=-1, keepdim=True)
                mask = torch.arange(p.size(1), device=p.device)[None,:] > cut
                sl[mask] = -float("inf")
                logits = torch.zeros_like(logits).scatter(1, si, sl)
            probs = torch.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1).squeeze(1)

        if no_repeat_ngram_size > 0 and violates(next_id):
            next_id = logits.argmax(dim=-1)

        x = torch.cat([x, next_id[:, None]], dim=1)
        update_seen(x)

        if t + 1 >= min_new_tokens:
            if stop_on_eos and next_id.item() == EOS: break
            if stop_at_punct:
                cont = sp.decode(x[0, start:].tolist()).strip()
                if cont and cont[-1:] in PUNCTS: break

    return sp.decode(x[0, start:].tolist()).strip()


In [5]:
def gen_factual(q: str):
    return generate(q, max_new_tokens=8, min_new_tokens=2,
                    temperature=0.1, top_p=0.9, rep_penalty=1.2,
                    no_repeat_ngram_size=4, stop_on_eos=False, stop_at_punct=True)

def gen_freeform(q: str):
    return generate(q, max_new_tokens=64, min_new_tokens=12,
                    temperature=0.7, top_p=0.95, rep_penalty=1.1,
                    no_repeat_ngram_size=4, stop_on_eos=False, stop_at_punct=True)

prompts = [
    "საქართველოს დედაქალაქია — ",
    "საქართველოს ვალუტაა — ",
    "„ვეფხისტყაოსნის“ ავტორია — ",
    "საქართველო მრავალი თვალსაზრისით არის ",
]

for p in prompts:
    print("Prompt:", p)
    print("Factual:", gen_factual(p))
    print("Freeform:", gen_freeform(p))
    print("-"*60)


Prompt: საქართველოს დედაქალაქია — 
Factual: როგორც ცნობილია ამ დროისთვის დაკავებულია ერთი პირი
Freeform: საქართველომ როკის გვირაბი გახსნა უნგრეთიდან ვილნიუსში შავი ზღვის ჩრდილო აღმოსავლეთით მდებარეობს და საქართველოს აკავშირებს ორი რეგიონის სტატუსიც აქვს და საერთო ეროვნული პოლიტიკის განსაზღვრაზე საკმაოდ დიდი ხანია ვლაპარაკობთ რა უნდა ვიცოდეთ თბილისისთვის ან როგორ გავაკეთოთ განსაკუთრებული აქცენტი მისი გეოგრაფიული მდებარეობის გამო და არა იმაზე თუ რომელი ქვეყნის ცხოვრობა შეიძლება
------------------------------------------------------------
Prompt: საქართველოს ვალუტაა — 
Factual: უძრავი ქონება ტრანსპორტზე გაცემული სესხების რაოდენობა
Freeform: უძრავი ქონება არ შეესაბამება რეალობას და არანაირად არ შეესაბამება საქართველოს ბანკის მიერ მიღებული ოფიციალური გაცვლითი კურსის მოთხოვნებსა და სტანდარტებს და მოთხოვნებს როგორც კომერციული ბანკები აცხადებენ ისინი ამ დრომდე გასცემენ სესხებს პირდაპირ უცხოურ ვალუტაში ან ლარში გასცემენ სხვადასხვა ლომბარდებს ან ლარში გასცემენ სესხს ნაღდი ფულის სახით ან იპოთეკურ

In [6]:
grid = []
q = "საქართველოს დედაქალაქია — "
for T in [0.0, 0.05, 0.1, 0.2]:
    for P in [0.8, 0.9, 0.95]:
        out = generate(q, temperature=T, top_p=P, max_new_tokens=8, min_new_tokens=2)
        grid.append({"temp": T, "top_p": P, "out": out})
grid


[{'temp': 0.0,
  'top_p': 0.8,
  'out': 'ამ ეტაპზე მიმდინარეობს მუშაობა იმ მიმართულებითაც'},
 {'temp': 0.0,
  'top_p': 0.9,
  'out': 'ამ ეტაპზე მიმდინარეობს მუშაობა იმ მიმართულებითაც'},
 {'temp': 0.0,
  'top_p': 0.95,
  'out': 'ამ ეტაპზე მიმდინარეობს მუშაობა იმ მიმართულებითაც'},
 {'temp': 0.05,
  'top_p': 0.8,
  'out': 'ამ ეტაპზე მიმდინარეობს მუშაობა იმ მიმართულებითაც'},
 {'temp': 0.05,
  'top_p': 0.9,
  'out': 'ამ ეტაპზე მიმდინარეობს მუშაობა იმ მიმართულებითაც'},
 {'temp': 0.05,
  'top_p': 0.95,
  'out': 'ამ ეტაპზე მიმდინარეობს მუშაობა იმ პირთა დადგენის'},
 {'temp': 0.1,
  'top_p': 0.8,
  'out': 'ამ ეტაპზე მიმდინარეობს მუშაობა იმ მიმართულებითაც'},
 {'temp': 0.1,
  'top_p': 0.9,
  'out': 'ამ ეტაპზე მიმდინარეობს მუშაობა იმ პირთა დადგენის'},
 {'temp': 0.1,
  'top_p': 0.95,
  'out': 'ამ ეტაპზე მიმდინარეობს მუშაობა იმ პირთა სია'},
 {'temp': 0.2, 'top_p': 0.8, 'out': 'თუმცა ამ ეტაპზე უცნობია თუ ვინ იქნება'},
 {'temp': 0.2, 'top_p': 0.9, 'out': 'მე არ ვიცი რა ხდება ამ დროს'},
 {'temp': 0.2,
 

In [7]:
examples = []
for p in prompts:
    examples.append({"prompt": p, "factual": gen_factual(p), "freeform": gen_freeform(p)})
Path("reports").mkdir(exist_ok=True, parents=True)
with open("reports/infer_examples.json", "w", encoding="utf-8") as f:
    json.dump(examples, f, ensure_ascii=False, indent=2)
examples


[{'prompt': 'საქართველოს დედაქალაქია — ',
  'factual': 'როგორც ცნობილია ამ დროისათვის დაკავებულია ორი ადამიანი',
  'freeform': 'აღსანიშნავია ის ფაქტიც რომ ამ წყვილმა ორივე ერთად დაამარცხა უკვე და თან დიდ სლემზე გასულა ზაფხულშიც რომ გაუშვია ბურთი ერთხელაც არ გაუმართლა ალბათ ყველაზე დიდი შეცდომა იყო რო გნახათ თუ არ იცით და მაგიტო იკნებაოოოოოოოოოოოოოოოოოოოოოოოოოოოოოოოოოოოო'},
 {'prompt': 'საქართველოს ვალუტაა — ',
  'factual': 'უძრავი ქონება ტრანსპორტზე გაცემული უცხოური ვალუტით',
  'freeform': 'სხვადასხვა ადგილიდან თუ პირადი შემადგენლიდან შემოსავალით მიღებული სახსრებიდან თანხის გადარიცხვას მიზანშეწონილად არ მიიჩნევს სახელმწიფო აუდიტის სამსახური საჯარო ინფორმაციის მოთხოვნის შესახებ ინფორმაციას ავრცელებს და შესაბამის განმარტებებსაც აქვეყნებს საქართველოს პროკურატურაში მიმდინარე გამოძიების ფარგლებში ჩატარებული საგამოძიებო მოქმედებების თაობაზე აკეთებს განცხადებას და საზოგადოებას აცნობებს ინფორმაციას აღნიშნულ საკითხთან დაკავშირებით შექმნილი მდგომარეობის შესახებ და ამ მიმართულებით დამატებით ინფორ